Extract Compressed GEE export folders, tif to main folder

In [1]:
import os
import zipfile

# Define the paths
source_folder = r"C:\Users\grupp\Desktop\Urugay NDVI"

def extract_tifs_from_zips(source_folder):
    # Iterate over all files in the source folder
    for file_name in os.listdir(source_folder):
        file_path = os.path.join(source_folder, file_name)
        
        # Check if the file is a zip file
        if zipfile.is_zipfile(file_path):
            with zipfile.ZipFile(file_path, 'r') as zip_ref:
                # List all files in the zip
                for member in zip_ref.namelist():
                    # Check if the file is a .tif file
                    if member.endswith('.tif'):
                        # Extract the .tif file to the source folder
                        zip_ref.extract(member, source_folder)
                        print(f"Extracted: {member} from {file_name}")

if __name__ == "__main__":
    extract_tifs_from_zips(source_folder)
    print("All .tif files have been extracted.")

Extracted: NDVI_2024_04_to_06-0000032768-0000032768.tif from drive-download-20250121T133606Z-002.zip
Extracted: NDVI_2024_07_to_09-0000000000-0000032768.tif from drive-download-20250121T133606Z-004.zip
Extracted: NDVI_2024_10_to_12-0000000000-0000032768.tif from drive-download-20250121T133606Z-006.zip
Extracted: NDVI_2024_07_to_09-0000032768-0000032768.tif from drive-download-20250121T133606Z-007.zip
Extracted: NDVI_2024_01_to_03-0000000000-0000032768.tif from drive-download-20250121T133606Z-011.zip
Extracted: NDVI_2024_01_to_03-0000032768-0000032768.tif from drive-download-20250121T133606Z-013.zip
Extracted: NDVI_2024_04_to_06-0000000000-0000032768.tif from drive-download-20250121T133606Z-015.zip
Extracted: NDVI_2023_10_to_12-0000032768-0000032768.tif from drive-download-20250121T133613Z-001.zip
Extracted: NDVI_2023_10_to_12-0000000000-0000032768.tif from drive-download-20250121T133613Z-002.zip
Extracted: NDVI_2023_04_to_06-0000032768-0000032768.tif from drive-download-20250121T133613

To do in arcpro: mosaic gee results and scale

In [ ]:
import arcpy
import os
import re

# Define the input folder and output folder
input_folder = r"C:\Users\grupp\Desktop\Urugay NDVI"
output_folder = r"C:\Users\grupp\Desktop\Urugay NDVI\Mosaicked"

temp_folder = os.path.join(output_folder, "Temp")
# Create the output and temporary folders if they don't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
if not os.path.exists(temp_folder):
    os.makedirs(temp_folder)

# Regular expression to extract the common prefix from file names
prefix_pattern = re.compile(r"^(.*?)-\d{10}-\d{10}-\d{3}\.tif$")

# Dictionary to group files by common prefix
grouped_files = {}

# Iterate through all .tif files in the input folder
for file_name in os.listdir(input_folder):
    if file_name.endswith(".tif"):
        match = prefix_pattern.match(file_name)
        if match:
            prefix = match.group(1)
            if prefix not in grouped_files:
                grouped_files[prefix] = []
            grouped_files[prefix].append(os.path.join(input_folder, file_name))

# Function to reclassify and scale a raster
def reclassify_and_scale(input_raster, output_raster):
    # Set all negative values to 0 and multiply by 10
    temp_raster = os.path.join(temp_folder, "temp_reclass.tif")
    arcpy.sa.Con(
        arcpy.sa.Raster(input_raster) < 0, 0, arcpy.sa.Raster(input_raster)
    ).save(temp_raster)
    scaled_raster = arcpy.sa.Times(temp_raster, 10)
    scaled_raster.save(output_raster)
    print(f"Reclassified and scaled: {input_raster} -> {output_raster}")

# Process files and mosaic
for prefix, file_list in grouped_files.items():
    reclassified_files = []

    for file_path in file_list:
        # Reclassify and scale each raster
        reclassified_raster = os.path.join(temp_folder, f"reclassified_{os.path.basename(file_path)}")
        reclassify_and_scale(file_path, reclassified_raster)
        reclassified_files.append(reclassified_raster)

    if len(reclassified_files) > 1:  # Only mosaic if there are multiple files
        output_raster = os.path.join(output_folder, f"{prefix}.tif")
        print(f"Mosaicking files for prefix '{prefix}' into {output_raster}")

        # Perform the mosaic
        arcpy.management.MosaicToNewRaster(
            input_rasters=reclassified_files,
            output_location=output_folder,
            raster_dataset_name_with_extension=f"{prefix}.tif",
            coordinate_system_for_the_raster=None,  # Use the default coordinate system
            pixel_type="8_BIT_UNSIGNED",
            cellsize=None,  # Use the default cell size
            number_of_bands=1,  # Assuming single-band rasters
            mosaic_method="LAST",
            mosaic_colormap_mode="MATCH"
        )

print("Processing and mosaicking complete.")